# 🏨 სასტუმროს AI ხმოვანი ასისტენტი — სრული გიდი LangGraph-ზე

მოგესალმებით! ეს ნოუთბუქი შექმნილია იმისთვის, რომ **ნულიდან** გაგარკვიოთ თანამედროვე AI აგენტების არქიტექტურაში.

ჩვენ ავაგებთ სასტუმროს ხმოვან ასისტენტს, რომელიც:
1. იგებს სტუმრის ქართულ მეტყველებას (STT);
2. მართავს საუბრის კონტექსტს და იღებს შეკვეთებს **LangGraph**-ის საშუალებით;
3. უერთდება სასტუმროს სისტემას (PMS Tools) და ამუშავებს შეცდომებს;
4. შეცდომის შემთხვევაში ასწორებს პასუხს ან ზარს გადაამისამართებს ადამიანთან (Human Escalation);
5. ოპტიმიზირებულია დაბალი დაყოვნებისთვის (Streaming Latency).

--- 
## 🧠 მოდული 1: თეორიული საფუძვლები (Zero-Knowledge Guide)

სანამ კოდს დავწერთ, გავიაზროთ 4 ძირითადი ცნება:

### 1. რა არის State (მდგომარეობა)?
წარმოიდგინეთ ოფიციანტი, რომელიც თქვენს მაგიდასთან მოდის ბლოკნოტით:
* სანამ თქვენ ელაპარაკებით, ის ბლოკნოტში იწერს: ოთახის ნომერს, რა შეუკვეთეთ და რა შეეშალა სამზარეულოს.
* **State არის სწორედ ეს ბლოკნოტი.** ჩვეულებრივი ჩატბოტი ყოველი შეკითხვის შემდეგ ივიწყებს ყველაფერს. LangGraph-ში State არის ერთი ცენტრალური მონაცემთა სტრუქტურა (ლექსიკონი), რომელსაც ყველა ეტაპი კითხულობს და ანახლებს.

### 2. რა არის Reducer (`add_messages`)?
თუ State-ში გვაქვს სია `messages = [msg1]`, და ახალმა კვანძმა დააბრუნა `messages = [msg2]`, ჩვეულებრივ პითონში ძველი წაიშლება და დარჩება მხოლოდ `[msg2]`.
`add_messages` ფუნქცია კი მუშაობს როგორც **append**: ის ძველ ისტორიას ავტომატურად ამატებს ახალ მესიჯს (`[msg1, msg2]`). ამას ქვია Reducer.

### 3. Node (კვანძი), Edge (ბმა) და Conditional Edge (პირობითი განშტოება)
* **Node (კვანძი):** ჩვეულებრივი Python ფუნქცია, რომელიც აკეთებს კონკრეტულ საქმეს (მაგ. ეკითხება LLM-ს ან ბაზაში ამოწმებს შეკვეთას).
* **Edge (ბმა):** პირდაპირი ისარი A კვანძიდან B კვანძამდე.
* **Conditional Edge (პირობითი განშტოება):** საგზაო გზაჯვარედინი. ლოგიკა, რომელიც State-ის მიხედვით წყვეტს: წავიდეთ `tools`-ზე, გადავამისამართოთ ადამიანთან (`human_escalation`) თუ დავასრულოთ საუბარი (`END`).

### 4. Tool Call და Error Handling (შეცდომის დამუშავება)
როცა LLM გადაწყვეტს, რომ შეკვეთა გააფორმოს, ის იძახებს Tool-ს. თუ ბაზამ დააბრუნა შეცდომა (მაგ. სასურველი კერძი მენიუში არ არის ან ქსელი გაითიშა), პროგრამა კი არ უნდა დაქრაშოს, არამედ შეცდომა უნდა ჩაიწეროს State-ში, რათა LLM-მა სტუმარს ალტერნატივა შესთავაზოს.

In [ ]:
# საჭირო ბიბლიოთეკების ინსტალაცია (საჭიროების შემთხვევაში)
# %pip install langgraph langchain-core langchain-google-genai python-dotenv

In [ ]:
import os
import getpass

# Gemini API გასაღების დაყენება (თუ გარემოს ცვლადებში არ გაქვთ)
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("შეიყვანეთ თქვენი Google Gemini API Key: ")

--- 
## 🛠️ მოდული 2: State-ის სქემა და სასტუმროს PMS Tool-ები

In [ ]:
from typing import Annotated, List, Optional, Dict, Any
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from langchain_core.tools import tool

# 1. STATE SCHEMA
class HotelAgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]  # საუბრის ისტორია (append რეჟიმში)
    room_number: Optional[str]                            # ოთახის ნომერი
    tool_error_count: int                                 # შეცდომების მთვლელი


# 2. MOCK PMS TOOLS (სასტუმროს სერვისების იმიტაცია)
ROOM_SERVICE_MENU = {
    "ყავა": 8.0,
    "ომლეტი": 15.0,
    "წყალი": 3.0,
    "კრუასანი": 7.0
}

@tool
def order_room_service(item: str, quantity: int = 1) -> str:
    """ნომერში საჭმლის ან სასმლის შეკვეთის ხელსაწყო."""
    item_clean = item.strip().lower()
    
    # ხელოვნური შეცდომის პროვოცირება ტესტირებისთვის
    if "ხინკალი" in item_clean:
        raise ValueError("შეცდომა: 'ხინკალი' არ არის საუზმის მენიუში!")
        
    if item_clean in ROOM_SERVICE_MENU:
        total = ROOM_SERVICE_MENU[item_clean] * quantity
        return f"შეკვეთა დადასტურებულია: {quantity}x {item_clean}. ჯამური ღირებულება: {total} ლარი."
    else:
        return f"პროდუქტი '{item}' არ გვაქვს. ხელმისაწვდომია მხოლოდ: {list(ROOM_SERVICE_MENU.keys())}"

@tool
def get_wifi_info() -> str:
    """სასტუმროს Wi-Fi ქსელის სახელის და პაროლის გაგება."""
    return "ქსელი: Hotel_Guest_5G, პაროლი: WelcomeToTbilisi2026"

tools = [order_room_service, get_wifi_info]
tools_by_name = {t.name: t for t in tools}
print("✅ State სქემა და Tool-ები წარმატებით შეიქმნა!")

--- 
## 🔄 მოდული 3: კვანძები (Nodes), განშტოებები (Edges) და გრაფის აწყობა

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, ToolMessage
from langgraph.graph import StateGraph, END

# LLM ინიციალიზაცია Tool Binding-ით
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0).bind_tools(tools)

SYSTEM_PROMPT = """შენ ხარ სასტუმროს თავაზიანი ქართულენოვანი ხმოვანი ასისტენტი.
ისაუბრე მოკლედ, ბუნებრივად და კონკრეტულად (რადგან შენი პასუხი ხმით უნდა წაიკითხოს სისტემამ).
თუ სტუმარი რაიმეს შეკვეთას ითხოვს, გამოიძახე შესაბამისი ხელსაწყო (Tool).
თუ ხელსაწყომ შეცდომა დააბრუნა, გაითვალისწინე შეცდომა და შესთავაზე ალტერნატივა."""

# --- 1. NODES (კვანძები) ---

def agent_node(state: HotelAgentState):
    """LLM იღებს გადაწყვეტილებას: უპასუხოს სტუმარს თუ გამოიძახოს Tool"""
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}

def tool_node(state: HotelAgentState):
    """Tool-ების გაშვება და შეცდომის უსაფრთხო დაჭერა (Safe Execution)"""
    last_message = state["messages"][-1]
    new_messages = []
    error_count = state.get("tool_error_count", 0)
    
    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        args = tool_call["args"]
        selected_tool = tools_by_name.get(tool_name)
        
        try:
            result = selected_tool.invoke(args)
            new_messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))
        except Exception as e:
            # შეცდომა არ აგდებს მთლიან სისტემას, არამედ აღირიცხება State-ში
            error_count += 1
            error_feedback = f"Tool Execution Failed: {str(e)}"
            new_messages.append(ToolMessage(content=error_feedback, tool_call_id=tool_call["id"]))
            
    return {"messages": new_messages, "tool_error_count": error_count}

def human_escalation_node(state: HotelAgentState):
    """როდესაც შეცდომა ორჯერ განმეორდა, საქმეში ერთვება ადამიანი"""
    escalation_message = SystemMessage(
        content="სისტემური შეფერხებაა. უთხარი სტუმარს: 'ბოდიშს გიხდით, სისტემური შეფერხების გამო თქვენს ზარს გადავამისამართებ ადმინისტრატორთან.'"
    )
    res = llm.invoke([escalation_message])
    return {"messages": [res]}

# --- 2. CONDITIONAL EDGES (პირობითი ლოგიკა) ---

def route_after_agent(state: HotelAgentState):
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "tools"
    return END

def route_after_tools(state: HotelAgentState):
    # თუ შეცდომების რაოდენობა გადაცდა ლიმიტს (>= 2), გადავდივართ ადამიანზე
    if state.get("tool_error_count", 0) >= 2:
        return "human_escalation"
    return "agent"

# --- 3. გრაფის აწყობა ---
builder = StateGraph(HotelAgentState)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_node("human_escalation", human_escalation_node)

builder.set_entry_point("agent")

builder.add_conditional_edges("agent", route_after_agent, {
    "tools": "tools",
    END: END
})
builder.add_conditional_edges("tools", route_after_tools, {
    "human_escalation": "human_escalation",
    "agent": "agent"
})
builder.add_edge("human_escalation", END)

hotel_graph = builder.compile()
print("🚀 LangGraph აგებულია და მზადაა!")

### 📊 გრაფის ვიზუალიზაცია
მოდით გამოვსახოთ ჩვენი შექმნილი workflow ვიზუალურად:

In [ ]:
from IPython.display import Image, display

try:
    display(Image(hotel_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(hotel_graph.get_graph().draw_ascii())

--- 
## 🧪 მოდული 4: ტესტირება რეალურ სცენარებზე

### სცენარი 1: წარმატებული Tool Call (Wi-Fi-ს გაგება)

In [ ]:
from langchain_core.messages import HumanMessage

state_input_1 = {
    "messages": [HumanMessage(content="გამარჯობა, Wi-Fi-ს პაროლი მითხარით რა.")],
    "room_number": "304",
    "tool_error_count": 0
}

output_1 = hotel_graph.invoke(state_input_1)

for msg in output_1["messages"]:
    print(f"[{msg.__class__.__name__}]: {msg.content}\n")

### სცენარი 2: Tool Failure და Self-Correction (თვითშესწორება)
სტუმარი ითხოვს „ხინკალს“. ჩვენმა Tool-მა იცის, რომ ხინკალი არ გვაქვს და ისვრის `ValueError`-ს. 
დააკვირდით, როგორ არ იქრაშება სისტემა და როგორ სთავაზობს LLM ალტერნატივას:

In [ ]:
state_input_2 = {
    "messages": [HumanMessage(content="ნომერში ხინკალი ამომიტანეთ სასწრაფოდ.")],
    "room_number": "304",
    "tool_error_count": 0
}

output_2 = hotel_graph.invoke(state_input_2)

for msg in output_2["messages"]:
    print(f"[{msg.__class__.__name__}]: {msg.content}\n")

print(f"შეცდომების რაოდენობა State-ში: {output_2['tool_error_count']}")

--- 
## ⚡ მოდული 5: Voice Pipeline & Latency Optimization (დაყოვნების შემცირება)

ხმოვან ასისტენტში მომხმარებელი ვერ დაელოდება 3-4 წამს, სანამ LLM მთელ წინადადებას ჩამოწერს და შემდეგ TTS დაიწყებს ლაპარაკს.

**გადაჭრა:** `Sentence-Level Streaming`
ჩვენ ვიღებთ LLM-ის ტოკენებს ნაკადად (`stream`), ვაგროვებთ ბუფერში და პირველივე სასვენ ნიშანზე (`.`, `!`, `?`) გამზადებულ წინადადებას მაშინვე ვგზავნით TTS-ში!

In [ ]:
import re
import time

def simulate_voice_streaming(prompt_text: str):
    print(f"🎙️ სტუმარი: {prompt_text}\n")
    
    input_state = {
        "messages": [HumanMessage(content=prompt_text)],
        "room_number": "304",
        "tool_error_count": 0
    }
    
    buffer = ""
    sentence_endings = re.compile(r'([.!?\n])')
    
    print("🔊 TTS აუდიო ნაკადის სიმულაცია (პირველივე წინადადება იკითხება მყისიერად):\n")
    
    # ვიყენებთ stream_mode="messages" რეალურ დროში ტოკენების წამოსაღებად
    for msg_chunk, metadata in hotel_graph.stream(input_state, stream_mode="messages"):
        if msg_chunk.content and metadata.get("langgraph_node") == "agent":
            text_piece = msg_chunk.content
            buffer += text_piece
            
            # ვამოწმებთ, დასრულდა თუ არა წინადადება
            parts = sentence_endings.split(buffer)
            if len(parts) > 1:
                # გვაქვს დასრულებული წინადადება
                complete_sentence = parts[0] + parts[1]
                buffer = "".join(parts[2:])  # ნარჩენს ვტოვებთ ბუფერში
                
                if complete_sentence.strip():
                    print(f"   🗣️ [TTS აჟღერებს წამიერად]: \"{complete_sentence.strip()}\"")
                    time.sleep(0.3)  # აუდიოს გაჟღერების დროითი სიმულაცია
                    
    if buffer.strip():
        print(f"   🗣️ [TTS ბოლო ფრაზა]: \"{buffer.strip()}\"")

# გავტესტოთ ხმოვანი ნაკადი
simulate_voice_streaming("დილა მშვიდობისა, ყავა მინდა ოთახში და როდის იქნება მზად?")

--- 
## 🎓 მოდული 6: მოსამზადებელი კითხვები ტექნიკური გასაუბრებისთვის

როდესაც ამ პროექტს წარადგენ გასაუბრებაზე, ზუსტად ამ დეტალებს ჩაგეძიებიან:

1. **რა მოხდებოდა State-ში `Annotated[..., add_messages]`-ის ნაცვლად ჩვეულებრივი `list` რომ გამოგვეყენებინა?**
   * *სწორი პასუხი:* ჩვეულებრივი სია ყოველ ახალ ნაბიჯზე გადაეწერებოდა (Overwrite). წინა საუბრის ისტორია დაიკარგებოდა და LLM ვეღარ მიხვდებოდა, რა კითხვა დასვა მომხმარებელმა ან რა პასუხი დააბრუნა Tool-მა. `add_messages` კი ამატებს ელემენტებს სიის ბოლოში (Append/Reducer).

2. **როგორ ავიცილეთ თავიდან მთლიანი პროექტის დაქრაშვა, როცა PMS Tool-მა ერორი დააბრუნა?**
   * *სწორი პასუხი:* `tool_node`-ში ჩავაშენეთ `try-except` ბლოკი. შეცდომის ტექსტი ჩავწერეთ `ToolMessage`-ში და გრაფი დავაბრუნეთ ისევ `agent`-თან. LLM-მა წაიკითხა შეცდომა და თავად გამოასწორა პასუხი სტუმართან.

3. **როგორ გავუმკლავდით Latency-ს (დაყოვნებას) ხმოვან რეჟიმში?**
   * *სწორი პასუხი:* არ ველოდებოდით მთლიანი პასუხის გენერაციას. გამოვიყენეთ `Token Streaming` და `Sentence-Level Chunking` (რეგულარული გამოსახულებებით `.`, `!`, `?` ნიშნებზე დაჭრა). პირველივე მოკლე წინადადება მაშინვე იგზავნებოდა Gemini TTS-ში, რაც Time-To-First-Audio-ს 3 წამიდან 600 მილიწამამდე ამცირებს.